In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time

In [2]:
START_DATE = "2022-01-01"
END_DATE = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")


AQI_FEATURES = [
    "pm10",
    "pm2_5",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone",
    "aerosol_optical_depth",
    "dust",
    "uv_index",
    "uv_index_clear_sky",
    "ammonia"
]

WEATHER_FEATURES = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "pressure_msl",
    "surface_pressure",
    "cloud_cover",
    "cloud_cover_low",
    "cloud_cover_mid",
    "cloud_cover_high",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
    "shortwave_radiation",
    "direct_radiation",
    "diffuse_radiation"
]


AQI_API = "https://air-quality-api.open-meteo.com/v1/air-quality"
WEATHER_API = "https://archive-api.open-meteo.com/v1/archive"


def fetch_city_data(city, latitude, longitude):

    # AQI API CALL
    aqi_params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": ",".join(AQI_FEATURES),
        "start_date": START_DATE,
        "end_date": END_DATE,
        "timezone": "auto"
    }

    aqi_response = requests.get(AQI_API, params=aqi_params).json()

    # WEATHER API CALL
    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": ",".join(WEATHER_FEATURES),
        "start_date": START_DATE,
        "end_date": END_DATE,
        "timezone": "auto"
    }

    weather_response = requests.get(
        WEATHER_API,
        params=weather_params
    ).json()

    # CREATE DATAFRAMES
    df_aqi = pd.DataFrame(aqi_response["hourly"])
    df_weather = pd.DataFrame(weather_response["hourly"])

    # MERGE
    df = pd.merge(df_aqi, df_weather, on="time")

    # ADD CITY
    df["city"] = city

    return df



In [3]:
karachi_df = fetch_city_data(
    city="Karachi",
    latitude=24.8607,
    longitude=67.0011
)

In [4]:
lahore_df = fetch_city_data(
    city="Lahore",
    latitude=31.5204,
    longitude=74.3587
)

In [5]:
OUTPUT_FILE = "pakistan_aqi_weather.xlsx"

with pd.ExcelWriter(OUTPUT_FILE) as writer:

    karachi_df.to_excel(
        writer,
        sheet_name="Karachi",
        index=False
    )

    lahore_df.to_excel(
        writer,
        sheet_name="Lahore",
        index=False
    )

print(f"Saved File: {OUTPUT_FILE}")

Saved File: pakistan_aqi_weather.xlsx
